In [ ]:
from datetime import datetime
import re
import json
import csv
from dataclasses import dataclass
from abc import ABC, abstractmethod

@dataclass
class Operation:
  type: str            # deposit / withdraw / interest
  amount: float
  datetime: datetime
  balance: float       # баланс после операции
  status: str          # success / fail
  
class TransactionLoader(ABC):
  # Читает файл и возвращает грязный список словарей без валидации

  @abstractmethod
  def load(self, path: str) -> list[dict]:
    pass

class JsonTransactionLoader(TransactionLoader):
  def load(self, path: str) -> list[dict]:
      with open(path, "r", encoding="utf-8") as f:
          data = json.load(f)
      if not isinstance(data, list):
          raise ValueError("Ожидался JSON-массив записей")
      return [dict(r) for r in data if isinstance(r, dict)]


class CsvTransactionLoader(TransactionLoader):
  def load(self, path: str) -> list[dict]:
      with open(path, "r", encoding="utf-8", newline="") as f:
          reader = csv.DictReader(f)
          return [dict(row) for row in reader]


def loader_for(path: str) -> TransactionLoader:
  # Фабрика: выбирает загрузчик по расширению
  lower = path.lower()
  if lower.endswith(".json"):
      return JsonTransactionLoader()
  if lower.endswith(".csv"):
      return CsvTransactionLoader()
  raise ValueError("Поддерживаются только .json и .csv")


# Проверяет «сырые» записи и превращает их в Operation.
class TransactionValidator:

  DATE_FORMATS = (
    "%Y-%m-%d %H:%M:%S",
    "%d/%m/%Y %H:%M",
    "%d/%m/%Y %H:%M:%S",
    "%Y-%m-%d",
  )
  
  REQUIRED_FIELDS = ("operation", "amount", "date", "status")

  # Набор допустимых операций задаётся снаружи.
  def __init__(self, valid_operations,
              account_number: str | None = None,
              account_type: str | None = None):
    self.valid_operations = frozenset(valid_operations)
    self.account_number = account_number
    self.account_type = account_type.lower() if account_type else None

  # приведение «сырых» значений
  @staticmethod
  def _normalize(raw: dict) -> dict:
    out = {}
    for key, value in raw.items():
        if isinstance(value, str):
            value = value.strip()
            if value == "" or value.lower() in ("null", "none", "nan"):
                value = None
        if key in ("amount", "balance_after") and value is not None:
            try:
                value = float(value)
            except (TypeError, ValueError):
                value = None
        if key in ("operation", "status", "account_type",
                    "account_number", "date") and value is not None:
            value = str(value).strip()
        out[key] = value
    return out

  @classmethod
  def _parse_date(cls, value):
    if not isinstance(value, str):
        return None
    for fmt in cls.DATE_FORMATS:
        try:
            return datetime.strptime(value, fmt)
        except ValueError:
            continue
    return None

  # поиск аккаунта
  def _matches_account(self, record: dict) -> bool:
    if self.account_number is not None:
        return record.get("account_number") == self.account_number
    if self.account_type is not None:
        return str(record.get("account_type", "")).lower() == self.account_type
    return True

  # проверка одной записи
  def _is_valid(self, record: dict) -> bool:
    for field in self.REQUIRED_FIELDS:
        if field not in record:
            return False
    if record.get("operation") not in self.valid_operations:
        return False
    if record.get("status") not in ("success", "fail"):
        return False
    amount = record.get("amount")
    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        return False
    if amount <= 0:
        return False
    if self._parse_date(record.get("date")) is None:
        return False
    return True

  # --- публичный API ---
  def clean(self, raw_records: list[dict]) -> list[Operation]:
    cleaned: list[Operation] = []
    for raw in raw_records:
        record = self._normalize(raw)
        if not self._matches_account(record):
            continue
        if not self._is_valid(record):
            continue
        cleaned.append(Operation(
            type=record["operation"],
            amount=float(record["amount"]),
            datetime=self._parse_date(record["date"]),
            balance=record.get("balance_after") or 0.0,
            status=record["status"],
        ))
    cleaned.sort(key=lambda op: op.datetime)
    return cleaned

class Account():
  _account_counter = 1000
  valid_operations = frozenset({"deposit", "withdraw"})

  def __init__(self, account_holder: str, balance: float = 0):
    if not self._is_valid_holder(account_holder):
      raise ValueError("Некорректное имя владельца")

    if balance < 0:
      raise ValueError("Начальный баланс не может быть отрицательным")

    self.account_type = "Account"
    self.holder = account_holder
    self.account_number = f"ACC-{Account._account_counter}"
    Account._account_counter += 1

    self._balance = float(balance)
    self.operations_history = []
  
  @staticmethod
  def _is_valid_holder(name: str) -> bool:
    if not isinstance(name, str):
        return False
    # «Имя Фамилия»: первая буква заглавная, остальные строчные кирриллица или латиница.
    pattern = r"^[А-ЯЁA-Z][а-яёa-z]+ [А-ЯЁA-Z][а-яёa-z]+$"
    return re.fullmatch(pattern, name) is not None

  def _add_operation(self, op_type: str, amount: float, status: str):
    self.operations_history.append({
      "type": op_type,
      "amount": amount,
      "datetime": datetime.now(),
      "balance": self._balance,
      "status": status
    })
  
  def deposit(self, amount: float):
    if amount < 0:
      raise ValueError("Сумма пополнения не может быть отрицательной")
    self._balance += amount
    self._add_operation("deposit", amount, "success")

  def withdraw(self, amount: float) -> bool:
    if amount <= 0:
      raise ValueError("Сумма снятия должна быть положительной")
    if amount > self._balance:
      self._add_operation("withdraw", amount, "fail")
      return False
    self._balance -= amount
    self._add_operation("withdraw", amount, "success")
    return True
  
  def get_balance(self):
    return self._balance
  
  def get_history(self):
    return [operation.copy() for operation in self.operations_history]

  def analyze_operations(
    self,
    n: int = 5,
    sort_by: str = "amount",        # "amount" | "datetime" | "both"
    reverse: bool = True,           # True — по убыванию, False — по возрастанию
    min_amount = 0                  # выбрать границу для крупных операций
):

    if n <= 0:
        return []

    if sort_by == "amount":
        key = lambda op: op["amount"]
    elif sort_by == "datetime":
        key = lambda op: op["datetime"]
    elif sort_by == "both":
        key = lambda op: (op["amount"], op["datetime"])
    else:
        raise ValueError("sort_by должен быть 'amount', 'datetime' или 'both'")

    ops = [op for op in self.operations_history if op["amount"] >= min_amount]
    ops = sorted(ops, key=key, reverse=reverse)
    result = ops[:n]

    for op in result:
        print(
            f"{op['datetime']:%Y-%m-%d %H:%M:%S} | "
            f"{op['type']:8} | "
            f"{op['amount']:.2f} | "
            f"баланс: {op['balance']:.2f} | "
            f"{op['status']}"
        )
    return result

  def make_validator(self, account_number: str | None = None) -> TransactionValidator:
    return TransactionValidator(
        valid_operations=self.valid_operations,
        account_type=self.account_type,
        account_number=account_number,
    )

class CheckingAccount(Account):
  account_type = "Checking"
  valid_operations = frozenset({"deposit", "withdraw"})


class SavingsAccount(Account):
  account_type = "Savings"
  valid_operations = frozenset({"deposit", "withdraw", "interest"})

  def apply_interest(self, rate: float) -> float:
    # значение в процентах
    if rate < 0:
      raise ValueError("Процентная ставка не может быть отрицательной")
    
    interest = self._balance * rate / 100
    if interest == 0:
      return 0.0

    return interest
  
  def withdraw(self, amount) -> bool:
    if amount <= 0:
      raise ValueError("Сумма снятия должна быть положительной")
    
    limit = self.get_balance() * 0.5
    if limit < amount:
      self._add_operation("withdraw", amount, "fail")
      return False
    
    self._balance -= amount
    self._add_operation("withdraw", amount, "success")
    return True

# Оркестратор. загрузка → очистка → применение для аккаунта
class TransactionImporter:
  def __init__(self, loader: TransactionLoader,
              validator: TransactionValidator):
    self.loader = loader
    self.validator = validator

  def import_into(self, account: Account, path: str) -> int:
    raw = self.loader.load(path)
    cleaned = self.validator.clean(raw)
    return account.apply_operations(cleaned)

import unittest

class TestAccount(unittest.TestCase):

  def setUp(self):
    # Сбрасываем счётчик перед каждым тестом, чтобы номера были предсказуемы
    Account._account_counter = 1000
      
  def test_init_negative_balance_raises(self):
    with self.assertRaises(ValueError):
      Account("Иван Петров", -10)
      
  def test_account_number_format(self):
    acc = Account("Иван Петров")
    self.assertRegex(acc.account_number, r"^ACC-\d{4}$")
      
  def test_account_counter_increments(self):
    acc1 = Account("Иван Петров")
    acc2 = Account("Anna Smith")
    self.assertEqual(acc1.account_number, "ACC-1000")
    self.assertEqual(acc2.account_number, "ACC-1001")

  def test_withdraw_negative_raises(self):
    acc = Account("Иван Петров", 100)
    with self.assertRaises(ValueError):
        acc.withdraw(-10)
    self.assertEqual(acc.get_balance(), 100.0)
    self.assertEqual(len(acc.operations_history), 0)
      
  def test_withdraw_zero_raises(self):
    acc = Account("Иван Петров", 100)
    with self.assertRaises(ValueError):
        acc.withdraw(0)
    self.assertEqual(acc.get_balance(), 100.0)
    self.assertEqual(len(acc.operations_history), 0)
      
  def test_operation_structure(self):
    acc = Account("Иван Петров", 100)
    acc.deposit(50)
    op = acc.operations_history[0]
    required_keys = {"type", "amount", "datetime", "balance", "status"}
    self.assertEqual(set(op.keys()), required_keys)
    self.assertIsInstance(op["datetime"], datetime)